In [1]:
from src.core.services.connection.databricks_connector import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-data-batch-ingestion")
conn.connect()

spark = conn.spark

print(" Connected to Spark!")
print(f"Spark Version: {spark.version}")

2026-10-05 11:25:21.494 | INFO     | src.core.services.connection.databricks_connector:connect:34 - SparkSession ready | app=yandex-data-batch-ingestion | version=4.2.0 | runtime=serverless-connect


 Connected to Spark!
Spark Version: 4.2.0


In [13]:
import os

catalog = os.environ.get("DATABRICKS_CATALOG", "yandex_go_dev")
raw_schema = f"{catalog}.raw_files"
volume_name = "landing"

print(f"🔧 Checking infrastructure for catalog [{catalog}]...")


volume_path = f"/Volumes/{catalog}/raw_files/{volume_name}"
print(f" Volume is ready at: {volume_path}")

🔧 Checking infrastructure for catalog [yandex_go_dev]...
 Volume is ready at: /Volumes/yandex_go_dev/raw_files/landing


In [2]:
from src.core.services.data_ingestion import UserBatchIngestionService, TaxiTripBatchIngestionService

user_service = UserBatchIngestionService()
user_service.run(spark)

taxi_service = TaxiTripBatchIngestionService()
taxi_service.run(spark)

2026-10-05 11:25:29.712 | INFO     | src.core.services.data_ingestion.base_ingestion:run:71 - Start data ingestion for yandex_go_dev.bronze.users
2026-10-05 11:25:29.713 | INFO     | src.core.services.data_ingestion.base_ingestion:extract:38 - Reading data [avro] from: /Volumes/yandex_go_dev/raw_files/landing/avro
2026-10-05 11:25:29.877 | INFO     | src.core.services.data_ingestion.base_ingestion:load:55 - Writing data in table Delta: yandex_go_dev.bronze.users
2026-10-05 11:25:41.831 | INFO     | src.core.services.data_ingestion.base_ingestion:load:67 - Successfully loaded in yandex_go_dev.bronze.users
2026-10-05 11:25:41.834 | INFO     | src.core.services.data_ingestion.base_ingestion:run:71 - Start data ingestion for yandex_go_dev.bronze.taxi
2026-10-05 11:25:41.835 | INFO     | src.core.services.data_ingestion.base_ingestion:extract:38 - Reading data [parquet] from: /Volumes/yandex_go_dev/raw_files/landing/parquet
2026-10-05 11:25:41.835 | INFO     | src.core.services.data_ingesti

In [15]:
users_count = spark.table(f"{catalog}.bronze.users").count()
taxi_count = spark.table(f"{catalog}.bronze.taxi").count()

print(f"📊 Count users in bronze.users: {users_count}")
print(f"📊 Count of trips in bronze.taxi: {taxi_count}")


display(
    spark.table(f"{catalog}.bronze.taxi").orderBy(
        "_ingested_at", ascending=False
    )
)

📊 Count users in bronze.users: 500
📊 Count of trips in bronze.taxi: 6000


DataFrame[id: string, user_id: string, driver_id: string, vendor_id: int, pickup_datetime: timestamp, dropoff_datetime: timestamp, passenger_count: bigint, trip_distance: double, rate_code_id: bigint, store_and_fwd_flag: string, pu_location_id: int, do_location_id: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, airport_fee: double, _rescued_data: string, _ingested_at: timestamp, _source_file: string]

In [7]:
import time

from src.core.services.data_ingestion import DriverStreamingIngestionService

driver_service = DriverStreamingIngestionService(
    include_phone_in_base_schema=False
)
stream_query = driver_service.run(spark)

time.sleep(5)


print(f"Is Stream Active? -> {stream_query.isActive}")
display(spark.table("yandex_go_dev.bronze.drivers"))


2026-10-03 18:03:12.544 | INFO     | src.core.services.data_ingestion.streaming_ingestion:run:72 - Initiating streaming ingestion pipeline for yandex_go_dev.bronze.drivers
2026-10-03 18:03:12.544 | INFO     | src.core.services.data_ingestion.streaming_ingestion:extract:29 - Starting Stream Reading [json] from: /Volumes/yandex_go_dev/raw_files/landing/json
2026-10-03 18:03:12.733 | INFO     | src.core.services.data_ingestion.streaming_ingestion:load:51 - Writing continuous stream to Delta table: yandex_go_dev.bronze.drivers
2026-10-03 18:03:21.602 | INFO     | src.core.services.data_ingestion.streaming_ingestion:load:66 - Stream batch processed successfully. Query ID: 1f71cc13-3561-4ccc-b613-6c8b194cc744


Is Stream Active? -> False


DataFrame[car_number: string, experience: int, id: string, name: string, rating: double, timestamp: timestamp, _rescued_data: string, _ingested_at: timestamp, _source_file: string]